# EQ‑Proof — production‑grade interactive notebook

This notebook is an **app‑like front‑end** for **EQ‑Proof**, a personal tool that:

- **Validates** numeric outputs against a JSON **constraint spec**
- **Repairs** outputs with **minimal change** (least‑squares style projections)
- Produces a **tamper‑evident proof artifact** and optional **cryptographic attestation**
- Runs **offline by default** (network egress can be disabled at runtime)

> **Scope / support:** This is a personal tool. The notebook is provided **as‑is** as a portfolio artifact; it is not a supported product.

---

## Quick start (demo)
1. Go to the **Interactive UI** cell and press **Run EQ‑Proof** (uses a built‑in demo spec).
2. Try different strategies (**Alt. projections**) and export a Markdown/PDF proof report.

## Spec format (high level)
A spec is JSON with:
- `variables`: ordered list of variable names
- `constraints`: list of constraint objects (`bounds`, `equality`, `sumcap`, `simplex`, `monotone`)
- `units` (optional): per‑variable unit strings (e.g., `"m"`, `"kg"`, `"s"`, `"Hz"`)

The engine is deliberately conservative: if a constraint cannot be safely interpreted, it fails loudly.


In [13]:
# config / utils
from __future__ import annotations

import os, time, uuid, platform, json, hashlib, base64
from dataclasses import dataclass
from typing import Dict, Any, List, Tuple, Optional, Iterable

import numpy as np

class EQProofError(Exception):
    """Base error for EQ‑Proof."""

class SpecError(EQProofError):
    """Spec validation / parsing error."""

class RepairError(EQProofError):
    """Repair algorithm could not satisfy constraints within limits."""

class Config:
    # Numerical tolerances
    EQUALITY_TOL: float = 1e-9
    SUM_TOL: float = 1e-9

    # Alternating projections (Dykstra‑lite)
    ALTPROJ_ITERS: int = 200
    ALTPROJ_TOL: float = 1e-10

    # Crypto defaults
    DEMO_HMAC_KEY: bytes = b"DEMO_KEY"
    # If you want deterministic Ed25519 keys for demos, set EQPROOF_ED25519_SEED_BASE64.

    DEFAULT_UNITS: Dict[str, str] = {}

def now_ms() -> int:
    return int(time.time() * 1000)

def run_meta() -> Dict[str, Any]:
    return {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "time_ms": now_ms(),
        "run_id": str(uuid.uuid4()),
    }

def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def canonical_json(obj: Any) -> bytes:
    """Stable JSON bytes (sorted keys, no whitespace) for hashing/signing."""
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")

def json_loads_strict(s: str) -> Any:
    """JSON loader that produces helpful errors."""
    try:
        return json.loads(s)
    except json.JSONDecodeError as e:
        raise EQProofError(f"Invalid JSON (line {e.lineno}, col {e.colno}): {e.msg}") from e

def log_step(report: Dict[str, Any], step: Dict[str, Any]) -> None:
    report.setdefault("steps", []).append(step)

def badge(ok: bool, text: str) -> str:
    return f"✅ {text}" if ok else f"❌ {text}"

def as_float(x: Any) -> float:
    try:
        return float(x)
    except Exception as e:
        raise EQProofError(f"Expected a number, got {type(x).__name__}: {x!r}") from e

def b64e(b: bytes) -> str:
    return base64.b64encode(b).decode("ascii")

def b64d(s: str) -> bytes:
    return base64.b64decode(s.encode("ascii"))


In [14]:
# no_net
# EQ‑Proof is designed to run offline. This cell optionally disables outbound network
# connections by monkey‑patching Python's socket constructors.
#
# Set EQPROOF_ALLOW_NET=1 to opt out (e.g., when fetching packages in a fresh env).

import socket

_ORIG_SOCKET = socket.socket
_ORIG_CREATE_CONN = getattr(socket, "create_connection", None)

class _NoNetSocket(socket.socket):
    def connect(self, *a, **k):
        raise RuntimeError("Outbound network disabled by EQ‑Proof (set EQPROOF_ALLOW_NET=1 to opt out)")
    def connect_ex(self, *a, **k):
        raise RuntimeError("Outbound network disabled by EQ‑Proof (set EQPROOF_ALLOW_NET=1 to opt out)")

def enforce_offline() -> None:
    if os.environ.get("EQPROOF_ALLOW_NET", "0") not in ("1", "true", "True"):
        socket.socket = _NoNetSocket  # type: ignore
        if _ORIG_CREATE_CONN is not None:
            def _deny(*a, **k):
                raise RuntimeError("Outbound network disabled by EQ‑Proof (set EQPROOF_ALLOW_NET=1 to opt out)")
            socket.create_connection = _deny  # type: ignore

def restore_network() -> None:
    socket.socket = _ORIG_SOCKET  # type: ignore
    if _ORIG_CREATE_CONN is not None:
        socket.create_connection = _ORIG_CREATE_CONN  # type: ignore

enforce_offline()


In [15]:
# spec
from dataclasses import dataclass, field

ALLOWED_CONSTRAINT_TYPES = {"bounds", "equality", "sumcap", "simplex", "monotone"}

@dataclass
class Spec:
    name: str
    version: str
    variables: List[str]
    constraints: List[Dict[str, Any]]
    probes: List[Dict[str, Any]] = field(default_factory=list)
    alternates: List[str] = field(default_factory=list)
    units: Dict[str, str] = field(default_factory=dict)

def _validate_spec_dict(d: Dict[str, Any]) -> None:
    for k in ("name","version","variables","constraints"):
        if k not in d:
            raise SpecError(f"Spec missing required field: {k!r}")
    if not isinstance(d["variables"], list) or not all(isinstance(v,str) and v for v in d["variables"]):
        raise SpecError("Spec.variables must be a non-empty list[str].")
    if len(set(d["variables"])) != len(d["variables"]):
        raise SpecError("Spec.variables contains duplicates.")
    if not isinstance(d["constraints"], list):
        raise SpecError("Spec.constraints must be a list.")
    for i,c in enumerate(d["constraints"]):
        if not isinstance(c, dict):
            raise SpecError(f"Constraint #{i} must be an object/dict.")
        t = c.get("type")
        if t not in ALLOWED_CONSTRAINT_TYPES:
            raise SpecError(f"Constraint #{i} has unsupported type {t!r}. Allowed: {sorted(ALLOWED_CONSTRAINT_TYPES)}")
    if "units" in d and not isinstance(d["units"], dict):
        raise SpecError("Spec.units must be an object mapping var->unit.")
    if "probes" in d and not isinstance(d["probes"], list):
        raise SpecError("Spec.probes must be a list of objects.")

def spec_from_dict(d: Dict[str, Any]) -> Spec:
    _validate_spec_dict(d)
    return Spec(
        name=str(d["name"]),
        version=str(d["version"]),
        variables=list(d["variables"]),
        constraints=list(d["constraints"]),
        probes=list(d.get("probes", [])),
        alternates=list(d.get("alternates", [])),
        units=dict(d.get("units", {})),
    )

def load_spec(path: str) -> Spec:
    with open(path, "r", encoding="utf-8") as f:
        d = json.load(f)
    return spec_from_dict(d)

def spec_hash(spec: Spec) -> str:
    d = {
        "name": spec.name,
        "version": spec.version,
        "variables": spec.variables,
        "constraints": spec.constraints,
        "probes": spec.probes,
        "alternates": spec.alternates,
        "units": spec.units,
    }
    return sha256_bytes(canonical_json(d))


In [16]:
# units
# Minimal dimensional analysis / conversion, designed for offline, deterministic behavior.
# Add tokens as needed; unknown units raise a clear error.

from typing import Tuple

DIM0 = (0,0,0,0,0,0,0)

BASE = {
    "": (DIM0, 1.0),
    "1": (DIM0, 1.0),

    "m": ((1,0,0,0,0,0,0), 1.0),
    "mm": ((1,0,0,0,0,0,0), 1e-3),
    "cm": ((1,0,0,0,0,0,0), 1e-2),
    "km": ((1,0,0,0,0,0,0), 1e3),

    "kg": ((0,1,0,0,0,0,0), 1.0),
    "g":  ((0,1,0,0,0,0,0), 1e-3),

    "s":  ((0,0,1,0,0,0,0), 1.0),
    "min":((0,0,1,0,0,0,0), 60.0),
    "h":  ((0,0,1,0,0,0,0), 3600.0),

    "%": (DIM0, 0.01),
    "pct": (DIM0, 0.01),
}

DERIVED = {"Hz": ((0,0,-1,0,0,0,0), 1.0)}

def _mul(a,b): return tuple(x+y for x,y in zip(a,b))
def _div(a,b): return tuple(x-y for x,y in zip(a,b))
def _pow(a,p): return tuple(x*p for x in a)

def _atom(tok: str) -> Tuple[Tuple[int,...], float]:
    if tok in BASE: return BASE[tok]
    if tok in DERIVED: return DERIVED[tok]
    raise EQProofError(f"Unknown unit token: {tok!r}")

def parse_unit(u: str) -> Tuple[Tuple[int,...], float]:
    """Parse unit strings like 'm', 'm/s', 'kg*m^2/s^2'."""
    if u is None: u = ""
    u = str(u).strip()
    if u == "" or u == "1":
        return BASE[""]
    u = u.replace(" ", "")
    num, *den = u.split("/")
    dim = DIM0
    fac = 1.0

    def parse_side(side: str, sign: int):
        nonlocal dim, fac
        for tok in filter(None, side.split("*")):
            if "^" in tok:
                base, p = tok.split("^", 1)
                p = int(p)
            else:
                base, p = tok, 1
            d, f = _atom(base)
            if sign == +1:
                dim = _mul(dim, _pow(d, p))
                fac *= f**p
            else:
                dim = _div(dim, _pow(d, p))
                fac /= f**p

    parse_side(num, +1)
    if den:
        parse_side("*".join(den), -1)

    return dim, fac

def convert(val: float, from_u: str, to_u: str) -> float:
    d1, f1 = parse_unit(from_u)
    d2, f2 = parse_unit(to_u)
    if d1 != d2:
        raise EQProofError(f"Incompatible units: {from_u!r} -> {to_u!r}")
    return (float(val) * f1) / f2

def coerce_inputs_to_spec_units(values: dict, spec_units: Dict[str,str]):
    """Accept either raw numbers or {'value':..., 'unit':...} objects."""
    steps = []
    out = dict(values)
    for k,u in spec_units.items():
        if isinstance(values.get(k), dict) and "value" in values[k] and "unit" in values[k]:
            v = as_float(values[k]["value"])
            from_u = str(values[k]["unit"])
            v2 = convert(v, from_u, u)
            out[k] = v2
            steps.append({"op":"unit_convert","var":k,"from":from_u,"to":u,"value_in":v,"value_out":v2})
    return out, steps


In [17]:
# constraints & repair (core math)
#
# Philosophy:
# - Keep the core operations deterministic and offline.
# - Prefer convex projections (closest point in L2) where possible.
# - For non-linear equalities: support safe 'solve_for' substitutions.
# - For coupled constraints: use alternating projections until convergence.

from typing import Callable
import sympy as sp

# ---------- Helpers: dict <-> vector ----------
def vec_from_values(spec: Spec, values: Dict[str, float]) -> np.ndarray:
    return np.array([as_float(values.get(v, 0.0)) for v in spec.variables], dtype=float)

def values_from_vec(spec: Spec, x: np.ndarray) -> Dict[str, float]:
    return {v: float(x[i]) for i, v in enumerate(spec.variables)}

def subset_indices(spec: Spec, vars_: List[str]) -> np.ndarray:
    idx = []
    vset = set(spec.variables)
    for v in vars_:
        if v not in vset:
            raise SpecError(f"Unknown variable in constraint: {v!r}")
        idx.append(spec.variables.index(v))
    return np.array(idx, dtype=int)

# ---------- Bounds projection ----------
def project_bounds(x: np.ndarray, lo: np.ndarray, hi: np.ndarray) -> np.ndarray:
    return np.minimum(np.maximum(x, lo), hi)

# ---------- Bounded simplex projection ----------
def project_bounded_simplex(y: np.ndarray, s: float, lo: np.ndarray, hi: np.ndarray, *,
                           max_iter: int = 200, tol: float = 1e-12) -> np.ndarray:
    """Project y onto {x: sum x = s, lo<=x<=hi}. Uses dual bisection on tau."""
    s = float(s)
    lo = lo.astype(float); hi = hi.astype(float)

    if s < lo.sum() - 1e-12 or s > hi.sum() + 1e-12:
        raise RepairError(f"Simplex target sum {s} is infeasible for bounds: [{lo.sum()}, {hi.sum()}].")

    tau_lo = -1e6
    tau_hi =  1e6

    def f(tau: float) -> float:
        return float(np.clip(y - tau, lo, hi).sum() - s)

    flo = f(tau_lo); fhi = f(tau_hi)
    expand = 0
    while flo < 0 and expand < 20:
        tau_lo *= 2; flo = f(tau_lo); expand += 1
    expand = 0
    while fhi > 0 and expand < 20:
        tau_hi *= 2; fhi = f(tau_hi); expand += 1

    for _ in range(max_iter):
        tau = 0.5 * (tau_lo + tau_hi)
        val = f(tau)
        if abs(val) <= tol:
            break
        if val > 0:
            tau_lo = tau
        else:
            tau_hi = tau

    x = np.clip(y - tau, lo, hi)
    err = s - x.sum()
    if abs(err) > 1e-9:
        free = (x > lo + 1e-12) & (x < hi - 1e-12)
        if free.any():
            x[free] += err / free.sum()
            x = np.clip(x, lo, hi)
    return x

# ---------- Isotonic regression (monotone projection) ----------
def project_isotonic(y: np.ndarray, *, increasing: bool = True, lo: Optional[np.ndarray] = None, hi: Optional[np.ndarray] = None) -> np.ndarray:
    """L2 projection of y onto monotone sequence (PAVA). Optional bounds applied after projection."""
    y = np.asarray(y, dtype=float)
    if not increasing:
        y2 = -y
        x2 = project_isotonic(y2, increasing=True, lo=None if lo is None else -hi, hi=None if hi is None else -lo)
        return -x2

    n = len(y)
    level = y.copy()
    weight = np.ones(n, dtype=float)
    start = np.arange(n)
    end = np.arange(n)

    i = 0
    while i < n - 1:
        if level[i] <= level[i+1] + 1e-15:
            i += 1
            continue
        tot_w = weight[i] + weight[i+1]
        tot_lvl = (weight[i]*level[i] + weight[i+1]*level[i+1]) / tot_w
        level[i] = tot_lvl
        weight[i] = tot_w
        end[i] = end[i+1]
        level = np.delete(level, i+1)
        weight = np.delete(weight, i+1)
        start = np.delete(start, i+1)
        end = np.delete(end, i+1)
        n -= 1
        if i > 0:
            i -= 1

    x = np.empty(len(y), dtype=float)
    for lvl, s0, e0 in zip(level, start, end):
        x[s0:e0+1] = lvl

    if lo is not None or hi is not None:
        lo2 = lo if lo is not None else np.full_like(x, -np.inf)
        hi2 = hi if hi is not None else np.full_like(x,  np.inf)
        x = np.clip(x, lo2, hi2)
    return x

# ---------- Equality support via SymPy ----------
def _sympy_symbols_for(values: Dict[str, float]) -> Dict[str, sp.Symbol]:
    return {k: sp.symbols(k, real=True) for k in values.keys()}

def equality_residual(expr_str: str, values: Dict[str, float]) -> float:
    """Return lhs-rhs evaluated at values for expr like 'Eq(x+y,cap)' (or any SymPy Eq)."""
    syms = {k: sp.symbols(k, real=True) for k in values.keys()}
    eq = sp.sympify(expr_str, locals={"Eq": sp.Eq, **syms})
    expr = sp.simplify(eq.lhs - eq.rhs)
    subs_map = {syms[k]: float(v) for k, v in values.items() if k in syms}
    val = expr.subs(subs_map)
    return float(sp.N(val))

def equality_solve_for(expr_str: str, target: str, values: Dict[str, float]) -> Optional[float]:
    syms = {k: sp.symbols(k, real=True) for k in set(values) | {target}}
    eq = sp.sympify(expr_str, locals={"Eq": sp.Eq, **syms})
    try:
        sol = sp.solve(eq, syms[target], dict=True)
    except Exception as e:
        raise RepairError(f"Failed to solve equality {expr_str!r} for {target!r}: {e}") from e
    if not sol:
        return None
    expr = sol[0][syms[target]]
    return float(sp.N(expr.subs({syms[k]: float(v) for k, v in values.items() if k in syms})))

# ---------- Constraint evaluators ----------
def collect_bounds(spec: Spec) -> Tuple[np.ndarray, np.ndarray]:
    lo = np.full(len(spec.variables), -np.inf, dtype=float)
    hi = np.full(len(spec.variables),  np.inf, dtype=float)
    for c in spec.constraints:
        if c.get("type") == "bounds":
            v = c["var"]
            i = spec.variables.index(v)
            if "lower" in c and c["lower"] is not None:
                lo[i] = max(lo[i], as_float(c["lower"]))
            if "upper" in c and c["upper"] is not None:
                hi[i] = min(hi[i], as_float(c["upper"]))
            if lo[i] > hi[i]:
                raise SpecError(f"Infeasible bounds for {v!r}: lower > upper.")
    return lo, hi

def evaluate_violations(spec: Spec, x: np.ndarray, lo: np.ndarray, hi: np.ndarray) -> List[Dict[str, Any]]:
    vals = values_from_vec(spec, x)
    vios: List[Dict[str, Any]] = []

    for i,v in enumerate(spec.variables):
        if x[i] < lo[i] - 1e-12:
            vios.append({"type":"bounds", "var":v, "kind":"lower", "value":float(x[i]), "lower":float(lo[i])})
        if x[i] > hi[i] + 1e-12:
            vios.append({"type":"bounds", "var":v, "kind":"upper", "value":float(x[i]), "upper":float(hi[i])})

    for c in spec.constraints:
        t = c.get("type")
        if t == "equality":
            expr = c["expr"]
            tol = float(c.get("tol", Config.EQUALITY_TOL))
            res = abs(equality_residual(expr, vals))
            if res > tol:
                vios.append({"type":"equality", "expr":expr, "residual":res, "tol":tol, "solve_for":c.get("solve_for")})
        elif t == "sumcap":
            vars_ = c["vars"]
            idx = subset_indices(spec, vars_)
            cap = c.get("cap")
            cap_var = c.get("cap_var")
            if cap_var is not None:
                cap = vals.get(cap_var)
            cap = float(cap)
            s = float(x[idx].sum())
            tol = float(c.get("tol", Config.SUM_TOL))
            if s > cap + tol:
                vios.append({"type":"sumcap", "vars":vars_, "sum":s, "cap":cap, "tol":tol})
        elif t == "simplex":
            vars_ = c["vars"]
            idx = subset_indices(spec, vars_)
            s_target = c.get("sum_to")
            if c.get("sum_var") is not None:
                s_target = vals.get(c["sum_var"])
            s_target = float(s_target)
            s = float(x[idx].sum())
            tol = float(c.get("tol", Config.SUM_TOL))
            if abs(s - s_target) > tol:
                vios.append({"type":"simplex", "vars":vars_, "sum":s, "sum_to":s_target, "tol":tol})
        elif t == "monotone":
            vars_ = c["vars"]
            idx = subset_indices(spec, vars_)
            increasing = (str(c.get("direction","increasing")).lower() != "decreasing")
            y = x[idx]
            ok = np.all(np.diff(y) >= -1e-12) if increasing else np.all(np.diff(y) <= 1e-12)
            if not ok:
                vios.append({"type":"monotone", "vars":vars_, "direction":"increasing" if increasing else "decreasing"})
    return vios


In [18]:
# diagnose / repair engine
from typing import Any

def _apply_equality_constraints(spec: Spec, x: np.ndarray, *, report: Dict[str, Any]) -> np.ndarray:
    vals = values_from_vec(spec, x)
    for c in spec.constraints:
        if c.get("type") != "equality":
            continue
        expr = c["expr"]
        tol = float(c.get("tol", Config.EQUALITY_TOL))
        res = abs(equality_residual(expr, vals))
        if res <= tol:
            continue
        target = c.get("solve_for")
        if not target:
            raise RepairError(f"Equality constraint requires 'solve_for' for safe repair: {expr!r}")
        before = float(vals.get(target, np.nan))
        new = equality_solve_for(expr, target, vals)
        if new is None or not np.isfinite(new):
            raise RepairError(f"Could not solve equality {expr!r} for {target!r}.")
        vals[target] = float(new)
        log_step(report, {"op":"equality_solve", "expr":expr, "target":target, "before":before, "after":float(new), "residual":res})
    return vec_from_values(spec, vals)

def _apply_sumcap(spec: Spec, x: np.ndarray, lo: np.ndarray, hi: np.ndarray, *, report: Dict[str, Any]) -> np.ndarray:
    vals = values_from_vec(spec, x)
    for c in spec.constraints:
        if c.get("type") != "sumcap":
            continue
        idx = subset_indices(spec, c["vars"])
        cap = c.get("cap")
        if c.get("cap_var") is not None:
            cap = vals.get(c["cap_var"])
        cap = float(cap)
        tol = float(c.get("tol", Config.SUM_TOL))
        s = float(x[idx].sum())
        if s <= cap + tol:
            continue
        before = x[idx].copy()
        lo_s = lo[idx].copy(); hi_s = hi[idx].copy()
        x[idx] = project_bounded_simplex(x[idx], cap, lo_s, hi_s)
        log_step(report, {"op":"sumcap_project", "vars":c["vars"], "sum_before":s, "cap":cap, "before":before.tolist(), "after":x[idx].tolist()})
    return x

def _apply_simplex(spec: Spec, x: np.ndarray, lo: np.ndarray, hi: np.ndarray, *, report: Dict[str, Any]) -> np.ndarray:
    vals = values_from_vec(spec, x)
    for c in spec.constraints:
        if c.get("type") != "simplex":
            continue
        idx = subset_indices(spec, c["vars"])
        s_target = c.get("sum_to")
        if c.get("sum_var") is not None:
            s_target = vals.get(c["sum_var"])
        s_target = float(s_target)
        tol = float(c.get("tol", Config.SUM_TOL))
        s = float(x[idx].sum())
        if abs(s - s_target) <= tol:
            continue
        before = x[idx].copy()
        x[idx] = project_bounded_simplex(x[idx], s_target, lo[idx], hi[idx])
        log_step(report, {"op":"simplex_project", "vars":c["vars"], "sum_before":s, "sum_to":s_target, "before":before.tolist(), "after":x[idx].tolist()})
    return x

def _apply_monotone(spec: Spec, x: np.ndarray, lo: np.ndarray, hi: np.ndarray, *, report: Dict[str, Any]) -> np.ndarray:
    for c in spec.constraints:
        if c.get("type") != "monotone":
            continue
        idx = subset_indices(spec, c["vars"])
        direction = str(c.get("direction", "increasing")).lower()
        increasing = (direction != "decreasing")
        before = x[idx].copy()
        x[idx] = project_isotonic(x[idx], increasing=increasing, lo=lo[idx], hi=hi[idx])
        log_step(report, {"op":"monotone_project", "vars":c["vars"], "direction":"increasing" if increasing else "decreasing",
                          "before":before.tolist(), "after":x[idx].tolist()})
    return x

def _apply_all_constraints_once(spec: Spec, x: np.ndarray, lo: np.ndarray, hi: np.ndarray, *, report: Dict[str, Any], order: List[str]) -> np.ndarray:
    if "bounds" in order:
        before = x.copy()
        x = project_bounds(x, lo, hi)
        if np.linalg.norm(x - before) > 0:
            log_step(report, {"op":"bounds_clip", "delta_l2":float(np.linalg.norm(x-before)), "max_delta":float(np.max(np.abs(x-before)))})
    if "monotone" in order:
        x = _apply_monotone(spec, x, lo, hi, report=report)
    if "simplex" in order:
        x = _apply_simplex(spec, x, lo, hi, report=report)
    if "sumcap" in order:
        x = _apply_sumcap(spec, x, lo, hi, report=report)
    if "equality" in order:
        x = _apply_equality_constraints(spec, x, report=report)
    before = x.copy()
    x = project_bounds(x, lo, hi)
    if np.linalg.norm(x - before) > 0:
        log_step(report, {"op":"bounds_clip_end", "delta_l2":float(np.linalg.norm(x-before)), "max_delta":float(np.max(np.abs(x-before)))})
    return x

def _compute_probes(spec: Spec, values: Dict[str, float]) -> List[Dict[str, Any]]:
    out = []
    if not spec.probes:
        return out
    syms = {k: sp.symbols(k, real=True) for k in values.keys()}
    for p in spec.probes:
        name = p.get("name","probe")
        expr = p.get("expr")
        if not expr:
            continue
        try:
            e = sp.sympify(expr, locals=syms)
            v = e.subs({syms[k]: float(values[k]) for k in values if k in syms})
            out.append({"name": name, "expr": expr, "value": float(sp.N(v))})
        except Exception as ex:
            out.append({"name": name, "expr": expr, "error": str(ex)})
    return out

def diagnose_and_repair(
    spec: Spec,
    values: Dict[str, Any],
    strategy: str = "altproj",
    *,
    spec_path: str = "",
    inputs_path: str = "",
    verbose: bool = True,
) -> Dict[str, Any]:
    started_ms = now_ms()
    report: Dict[str, Any] = {
        "violations": [],
        "steps": [],
        "meta": {"env": run_meta(), "strategy": strategy, "spec_hash": spec_hash(spec)},
    }

    coerced, unit_steps = coerce_inputs_to_spec_units(values, getattr(spec, "units", Config.DEFAULT_UNITS))
    report["steps"].extend(unit_steps)

    x0 = vec_from_values(spec, coerced)
    lo, hi = collect_bounds(spec)

    if strategy == "bounds_then_equality":
        order = ["bounds", "equality", "monotone", "simplex", "sumcap"]
        x = _apply_all_constraints_once(spec, x0.copy(), lo, hi, report=report, order=order)
    elif strategy == "equality_only":
        x = _apply_equality_constraints(spec, x0.copy(), report=report)
    elif strategy == "altproj":
        order = ["bounds", "monotone", "simplex", "sumcap", "equality"]
        x = x0.copy()
        for it in range(int(Config.ALTPROJ_ITERS)):
            prev = x.copy()
            x = _apply_all_constraints_once(spec, x, lo, hi, report=report, order=order)
            delta = float(np.max(np.abs(x - prev)))
            if delta <= Config.ALTPROJ_TOL:
                log_step(report, {"op":"converged", "iter":it, "max_abs_delta":delta})
                break
        else:
            raise RepairError(f"Alternating projections did not converge in {Config.ALTPROJ_ITERS} iterations.")
    else:
        raise EQProofError(f"Unknown strategy: {strategy!r}")

    vios = evaluate_violations(spec, x, lo, hi)
    report["violations"] = vios
    report["meta"]["elapsed_ms"] = now_ms() - started_ms
    report["meta"]["spec_path"] = spec_path
    report["meta"]["inputs_path"] = inputs_path
    report["meta"]["l2_change"] = float(np.linalg.norm(x - x0))
    report["meta"]["max_abs_change"] = float(np.max(np.abs(x - x0))) if len(x) else 0.0

    original = values_from_vec(spec, x0)
    repaired  = values_from_vec(spec, x)

    probes = _compute_probes(spec, repaired)
    if probes:
        report["meta"]["probes"] = probes

    return {"original": original, "repaired": repaired, "report": report}

last_spec: Optional[Spec] = None
last_inputs: Optional[Dict[str, Any]] = None
last_result: Optional[Dict[str, Any]] = None
last_attestation: Optional[Dict[str, Any]] = None


In [19]:
# attest / verify
import hmac, hashlib
import nacl.signing
import nacl.exceptions

def _attestable_payload(spec_obj: Dict[str, Any], result: Dict[str, Any], *, spec_path: str, inputs_path: str) -> Dict[str, Any]:
    return {"spec": spec_obj, "result": result, "meta": {"spec_path": spec_path, "inputs_path": inputs_path}}

def attest_hmac(payload: Dict[str, Any], key: bytes) -> Dict[str, Any]:
    sig = hmac.new(key, canonical_json(payload), hashlib.sha256).digest()
    return {"algo":"HMAC-SHA256","signature_b64": b64e(sig)}

def verify_hmac(att: Dict[str, Any], payload: Dict[str, Any], key: bytes) -> bool:
    sig = b64d(att["signature_b64"])
    expect = hmac.new(key, canonical_json(payload), hashlib.sha256).digest()
    return hmac.compare_digest(sig, expect)

def ed25519_generate_keypair() -> Tuple[str, str]:
    sk = nacl.signing.SigningKey.generate()
    vk = sk.verify_key
    return (b64e(bytes(sk)), b64e(bytes(vk)))

def attest_ed25519(payload: Dict[str, Any], signing_key_b64: Optional[str] = None) -> Dict[str, Any]:
    msg = canonical_json(payload)
    if signing_key_b64 is None:
        seed_b64 = os.environ.get("EQPROOF_ED25519_SEED_BASE64")
        if seed_b64:
            seed = b64d(seed_b64)
            if len(seed) != 32:
                raise EQProofError("EQPROOF_ED25519_SEED_BASE64 must decode to 32 bytes.")
            sk = nacl.signing.SigningKey(seed)
        else:
            sk = nacl.signing.SigningKey.generate()
    else:
        sk = nacl.signing.SigningKey(b64d(signing_key_b64))
    sig = sk.sign(msg).signature
    return {"algo":"Ed25519","signature_b64": b64e(sig), "public_key_b64": b64e(bytes(sk.verify_key))}

def verify_ed25519(att: Dict[str, Any], payload: Dict[str, Any]) -> bool:
    try:
        pk = b64d(att["public_key_b64"])
        sig = b64d(att["signature_b64"])
        nacl.signing.VerifyKey(pk).verify(canonical_json(payload), sig)
        return True
    except nacl.exceptions.BadSignatureError:
        return False

def attest(spec_obj: Dict[str, Any], result: Dict[str, Any], *, spec_path: str, inputs_path: str, algo: str = "HMAC-SHA256") -> Dict[str, Any]:
    payload = _attestable_payload(spec_obj, result, spec_path=spec_path, inputs_path=inputs_path)
    if algo == "HMAC-SHA256":
        key = os.environ.get("EQPROOF_HMAC_KEY")
        key_b = Config.DEMO_HMAC_KEY if key is None else key.encode("utf-8")
        att = attest_hmac(payload, key_b)
    elif algo == "Ed25519":
        sk_b64 = os.environ.get("EQPROOF_ED25519_SIGNING_KEY_BASE64")
        att = attest_ed25519(payload, signing_key_b64=sk_b64)
    else:
        raise EQProofError(f"Unknown attestation algo: {algo}")
    att["payload_sha256"] = sha256_bytes(canonical_json(payload))
    return att

def verify_attestation(att: Dict[str, Any], spec_obj: Dict[str, Any], result: Dict[str, Any], *, spec_path: str, inputs_path: str) -> bool:
    payload = _attestable_payload(spec_obj, result, spec_path=spec_path, inputs_path=inputs_path)
    if att.get("payload_sha256") != sha256_bytes(canonical_json(payload)):
        return False
    algo = att.get("algo")
    if algo == "HMAC-SHA256":
        key = os.environ.get("EQPROOF_HMAC_KEY")
        key_b = Config.DEMO_HMAC_KEY if key is None else key.encode("utf-8")
        return verify_hmac(att, payload, key_b)
    if algo == "Ed25519":
        return verify_ed25519(att, payload)
    return False


In [20]:
# report / exports (Markdown + PDF)
import datetime
from pathlib import Path

try:
    from reportlab.lib.pagesizes import letter
    from reportlab.pdfgen import canvas as rl_canvas
    REPORTLAB_OK = True
except Exception:
    REPORTLAB_OK = False

def _fmt_num(x: Any) -> str:
    try:
        fx = float(x)
        if abs(fx) >= 1e6 or (abs(fx) > 0 and abs(fx) < 1e-6):
            return f"{fx:.6g}"
        return f"{fx:.6f}".rstrip("0").rstrip(".")
    except Exception:
        return str(x)

def render_markdown(spec_path: str, inputs_path: str, result: Dict[str, Any], attestation: Dict[str, Any], *, verbose: bool = True) -> str:
    ts = datetime.datetime.utcnow().isoformat() + "Z"
    o = result["original"]; r = result["repaired"]
    rep = result["report"]
    vios = rep.get("violations", [])
    probes = rep.get("meta", {}).get("probes", [])

    lines: List[str] = []
    lines.append("# EQ‑Proof report")
    lines.append(f"- Generated: {ts}")
    lines.append(f"- Spec: `{spec_path}`")
    lines.append(f"- Inputs: `{inputs_path}`")
    lines.append(f"- Strategy: `{rep.get('meta',{}).get('strategy')}`")
    lines.append(f"- Change (L2): `{_fmt_num(rep.get('meta',{}).get('l2_change'))}`")
    lines.append(f"- Max abs change: `{_fmt_num(rep.get('meta',{}).get('max_abs_change'))}`")
    lines.append("")

    keys = list(sorted(set(o) | set(r)))
    if keys:
        w = max(len(k) for k in keys)
        lines.append("## Original vs repaired")
        lines.append("```text")
        for k in keys:
            lines.append(f"{k:<{w}}  {_fmt_num(o.get(k))}  ->  {_fmt_num(r.get(k))}")
        lines.append("```")
        lines.append("")

    lines.append("## Remaining violations")
    if not vios:
        lines.append("- none")
    else:
        for v in vios[:50]:
            lines.append(f"- {json.dumps(v, ensure_ascii=False)}")
        if len(vios) > 50:
            lines.append(f"- ... ({len(vios)-50} more)")
    lines.append("")

    if probes:
        lines.append("## Probes")
        for p in probes:
            if "error" in p:
                lines.append(f"- {p['name']}: error {p['error']}")
            else:
                lines.append(f"- {p['name']}: {_fmt_num(p.get('value'))}  (expr: `{p.get('expr')}`)")
        lines.append("")

    if verbose:
        lines.append("## Repair steps")
        steps = rep.get("steps", [])
        lines.append("- none" if not steps else "")
        for s in steps:
            lines.append(f"- {json.dumps(s, ensure_ascii=False)}")
        lines.append("")
        lines.append("## Attestation")
        lines.append(f"- algo: `{attestation.get('algo')}`")
        if attestation.get("algo") == "Ed25519":
            lines.append(f"- public_key_b64: `{attestation.get('public_key_b64','')}`")
        lines.append(f"- signature_b64: `{attestation.get('signature_b64','')}`")
        lines.append(f"- payload_sha256: `{attestation.get('payload_sha256','')}`")
        lines.append("")

    return "\n".join(lines)

def save_pdf_from_lines(lines: List[str], out_path: str, *, title: str = "EQ‑Proof report") -> None:
    if not REPORTLAB_OK:
        raise EQProofError("reportlab not available; cannot export PDF.")
    c = rl_canvas.Canvas(out_path, pagesize=letter)
    width, height = letter
    x = 48
    y = height - 54
    c.setFont("Helvetica-Bold", 14)
    c.drawString(x, y, title)
    y -= 24
    c.setFont("Courier", 9)
    for line in lines:
        if y < 54:
            c.showPage()
            y = height - 54
            c.setFont("Courier", 9)
        c.drawString(x, y, line[:120].replace("`",""))
        y -= 12
    c.save()

def report_lines(spec_path: str, inputs_path: str, result: Dict[str, Any], attestation: Dict[str, Any]) -> List[str]:
    return [ln.replace("`","") for ln in render_markdown(spec_path, inputs_path, result, attestation, verbose=True).splitlines()]


In [21]:
# interactive UI
try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown, FileLink
    UI_AVAILABLE = True
except Exception:
    UI_AVAILABLE = False

DEMO_SPEC_DICT = {
    "name": "DemoSpec",
    "version": "1.0",
    "variables": ["p1","p2","p3","cap","x","y"],
    "constraints": [
        {"type":"bounds","var":"p1","lower":0,"upper":1},
        {"type":"bounds","var":"p2","lower":0,"upper":1},
        {"type":"bounds","var":"p3","lower":0,"upper":1},
        {"type":"bounds","var":"cap","lower":0,"upper":3},
        {"type":"bounds","var":"x","lower":0,"upper":10},
        {"type":"bounds","var":"y","lower":0,"upper":10},
        {"type":"monotone","vars":["p1","p2","p3"],"direction":"increasing"},
        {"type":"simplex","vars":["p1","p2","p3"],"sum_var":"cap","tol":1e-9},
        {"type":"sumcap","vars":["x","y"],"cap_var":"cap","tol":1e-9},
        {"type":"equality","expr":"Eq(x+y,cap)","solve_for":"y","tol":1e-9},
    ],
    "probes": [{"name":"p_sum","expr":"p1+p2+p3"},{"name":"xy_sum","expr":"x+y"}],
    "units": {}
}
DEMO_INPUTS = {"p1": 0.9, "p2": 0.2, "p3": -0.4, "cap": 1.0, "x": 2.0, "y": 10.0}

if not UI_AVAILABLE:
    print("ipywidgets not available in this runtime. Use the fallback demo cell below.")
else:
    spec_area = widgets.Textarea(value=json.dumps(DEMO_SPEC_DICT, indent=2), layout=widgets.Layout(width="100%", height="220px"))
    inputs_area = widgets.Textarea(value=json.dumps(DEMO_INPUTS, indent=2), layout=widgets.Layout(width="100%", height="140px"))

    strategy_dd = widgets.Dropdown(options=[("Alternating projections","altproj"),("Bounds then equality","bounds_then_equality"),("Equality only","equality_only")], value="altproj", description="Strategy")
    att_algo = widgets.Dropdown(options=["HMAC-SHA256","Ed25519"], value="HMAC-SHA256", description="Attest")
    iters = widgets.BoundedIntText(value=Config.ALTPROJ_ITERS, min=1, max=5000, step=10, description="Max iters")
    tol = widgets.FloatText(value=Config.ALTPROJ_TOL, description="Tol")

    run_btn = widgets.Button(description="Run EQ‑Proof", button_style="primary")
    export_md_btn = widgets.Button(description="Export MD")
    export_pdf_btn = widgets.Button(description="Export PDF")
    verify_btn = widgets.Button(description="Verify")

    status_out = widgets.Output()
    main_out = widgets.Output()
    export_out = widgets.Output()

    def _spec_obj(spec: Spec) -> Dict[str, Any]:
        return {"name": spec.name, "version": spec.version, "variables": spec.variables, "constraints": spec.constraints,
                "probes": spec.probes, "alternates": spec.alternates, "units": spec.units}

    def on_run(_):
        status_out.clear_output(); main_out.clear_output(); export_out.clear_output()
        try:
            Config.ALTPROJ_ITERS = int(iters.value)
            Config.ALTPROJ_TOL = float(tol.value)
            spec = spec_from_dict(json_loads_strict(spec_area.value))
            inputs = json_loads_strict(inputs_area.value)

            global last_spec, last_inputs, last_result, last_attestation
            last_spec, last_inputs = spec, inputs
            res = diagnose_and_repair(spec, inputs, strategy=strategy_dd.value, spec_path="ui_spec.json", inputs_path="ui_inputs.json")
            last_result = res
            att = attest(_spec_obj(spec), res, spec_path="ui_spec.json", inputs_path="ui_inputs.json", algo=att_algo.value)
            last_attestation = att

            md = render_markdown("ui_spec.json", "ui_inputs.json", res, att, verbose=True)
            with status_out:
                ok = len(res["report"].get("violations", [])) == 0
                print(badge(ok, "PASS" if ok else "FAIL"))
                print(f"Violations: {len(res['report'].get('violations', []))}")
            with main_out:
                display(Markdown(md))
        except Exception as e:
            with status_out:
                print(badge(False, f"Run failed: {e}"))

    def on_verify(_):
        export_out.clear_output()
        if last_spec is None or last_result is None or last_attestation is None:
            with export_out: print("Run first."); return
        ok = verify_attestation(last_attestation, _spec_obj(last_spec), last_result, spec_path="ui_spec.json", inputs_path="ui_inputs.json")
        with export_out:
            print(badge(ok, "Attestation verified" if ok else "Attestation invalid"))

    def on_export_md(_):
        export_out.clear_output()
        if last_result is None or last_attestation is None:
            with export_out: print("Run first."); return
        p = Path("eqproof_report.md")
        p.write_text(render_markdown("ui_spec.json","ui_inputs.json", last_result, last_attestation, verbose=True), encoding="utf-8")
        with export_out:
            print(badge(True, f"Wrote {p}"))
            try: display(FileLink(str(p)))
            except Exception: pass

    def on_export_pdf(_):
        export_out.clear_output()
        if last_result is None or last_attestation is None:
            with export_out: print("Run first."); return
        p = Path("eqproof_report.pdf")
        save_pdf_from_lines(report_lines("ui_spec.json","ui_inputs.json", last_result, last_attestation), str(p))
        with export_out:
            print(badge(True, f"Wrote {p}"))
            try: display(FileLink(str(p)))
            except Exception: pass

    run_btn.on_click(on_run)
    verify_btn.on_click(on_verify)
    export_md_btn.on_click(on_export_md)
    export_pdf_btn.on_click(on_export_pdf)

    display(widgets.VBox([
        widgets.HTML("<h3>EQ‑Proof Interactive UI</h3>"),
        widgets.HTML("<b>Spec JSON</b>"),
        spec_area,
        widgets.HTML("<b>Inputs JSON</b>"),
        inputs_area,
        widgets.HBox([strategy_dd, att_algo, iters, tol]),
        widgets.HBox([run_btn, verify_btn, export_md_btn, export_pdf_btn]),
        status_out,
        main_out,
        export_out,
    ]))


In [22]:
# fallback demo (no ipywidgets needed)
demo_spec = spec_from_dict(DEMO_SPEC_DICT)
demo_inputs = dict(DEMO_INPUTS)

res = diagnose_and_repair(demo_spec, demo_inputs, strategy="altproj", spec_path="demo_spec.json", inputs_path="demo_inputs.json")
spec_obj = {"name": demo_spec.name, "version": demo_spec.version, "variables": demo_spec.variables, "constraints": demo_spec.constraints,
            "probes": demo_spec.probes, "alternates": demo_spec.alternates, "units": demo_spec.units}
att = attest(spec_obj, res, spec_path="demo_spec.json", inputs_path="demo_inputs.json", algo="HMAC-SHA256")

print("Original:", res["original"])
print("Repaired :", res["repaired"])
print("Violations:", len(res["report"].get("violations", [])))
print("Attestation verified:", verify_attestation(att, spec_obj, res, spec_path="demo_spec.json", inputs_path="demo_inputs.json"))


# Notes
# - Projections: bounds (clip), simplex (bounded simplex projection), monotone (PAVA isotonic regression).
# - Equalities: repaired only via explicit 'solve_for' substitutions for safety.
# - Coupled constraints: alternating projections with configurable iters/tol.
# - Attestation: HMAC-SHA256 (local) or Ed25519 (public-key).
